In [1]:
from onep import paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

plt.rcParams.update({
    "font.size": 8,
    "axes.linewidth": 1.0,
    "font.family": "Arial"
})
sns.set(style="ticks", font="Arial")

# Load data
path_A = paths.processed("modeling", "astrocyte_NEWcxtA_Gaussian_noise3_all.csv")
path_B = paths.processed("modeling", "astrocyte_NEWcxtB_Gaussian_noise3_all.csv")
path_FC = paths.processed("modeling", "astrocyte_shot_Gaussian_noise3_all.csv")

df_A = pd.read_csv(path_A)
df_B = pd.read_csv(path_B)
df_FC = pd.read_csv(path_FC)

#create context column
df_A["context"] = "CxtA"
df_B["context"] = "CxtB"
df_FC["context"] = "FC"

# assign sex from animal string (F/M)
for d in (df_A, df_B, df_FC):
    d["sex"] = d["animal"].str.upper().str.contains("F").map({True: "F", False: "M"})

# set variance = NaN when 0 or non-numeric
for d in (df_A, df_B, df_FC):
    d["variance"] = pd.to_numeric(d["variance"], errors="coerce")
    d.loc[d["variance"] == 0, "variance"] = np.nan

# combined cell-level dataframe
df_all = pd.concat([df_A, df_B, df_FC], ignore_index=True)
df_all = df_all.dropna(subset=["variance", "animal", "context", "sex"])

print("Cell counts per context, sex:")
print(df_all.groupby(["context", "sex"])["variance"].count())

Cell counts per context, sex:
context  sex
CxtA     F       535
         M       819
CxtB     F       227
         M       596
FC       F      1074
         M      1672
Name: variance, dtype: int64


In [2]:

# Mouse-level 6-group plot (FC-F, FC-M, CxtA-F, CxtA-M, CxtB-F, CxtB-M)
mouse_avg = (
    df_all.groupby(["animal", "sex", "context"])["variance"]
    .mean()
    .dropna()
    .reset_index()
)

FC_F = mouse_avg[(mouse_avg["context"] == "FC") & (mouse_avg["sex"] == "F")]["variance"]
FC_M = mouse_avg[(mouse_avg["context"] == "FC") & (mouse_avg["sex"] == "M")]["variance"]
A_F = mouse_avg[(mouse_avg["context"] == "CxtA") & (mouse_avg["sex"] == "F")]["variance"]
A_M = mouse_avg[(mouse_avg["context"] == "CxtA") & (mouse_avg["sex"] == "M")]["variance"]
B_F = mouse_avg[(mouse_avg["context"] == "CxtB") & (mouse_avg["sex"] == "F")]["variance"]
B_M = mouse_avg[(mouse_avg["context"] == "CxtB") & (mouse_avg["sex"] == "M")]["variance"]

groups_mouse = [FC_F, FC_M, A_F, A_M, B_F, B_M]
labels_mouse = ["FC-F", "FC-M", "CxtA-F", "CxtA-M", "CxtB-F", "CxtB-M"]

fc_color = "#B39BC8"
cxtA_color = "#66C1A6"
cxtB_color = "black"

colors_mouse = [fc_color, fc_color, cxtA_color, cxtA_color, cxtB_color, cxtB_color]
hatches = ["", "//", "", "//", "", "//"]
positions = np.arange(1, 7)

fig, ax = plt.subplots(figsize=(4.5, 3.5), dpi=600)

bp = ax.boxplot(
    groups_mouse,
    positions=positions,
    widths=0.6,
    labels=labels_mouse,
    patch_artist=True
)

for patch, color, hatch in zip(bp["boxes"], colors_mouse, hatches):
    patch.set_facecolor(color)
    patch.set_edgecolor("black")
    patch.set_alpha(0.5)
    patch.set_hatch(hatch)

np.random.seed(0)
jitter = 0.10

for x_pos, vals, color in zip(positions, groups_mouse, colors_mouse):
    vals = np.array(vals)
    if len(vals) == 0:
        continue
    x = x_pos + np.random.uniform(-jitter, jitter, size=len(vals))
    ax.scatter(
        x, vals,
        s=35,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        zorder=3
    )

ax.set_ylabel("Mean variance per mouse")
ax.tick_params(axis="x", rotation=20)
sns.despine(ax=ax)
plt.tight_layout(pad=2)
plt.show()

C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2316875303.py:29: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2316875303.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

sns.set(style="ticks", font="Arial")
plt.rcParams.update({'font.size': 8})

# Sex colors
sex_palette = {
    "F": "#e06868",   # Female
    "M": "#5b7c86"    # Male
}

# Group order
order_6 = ["FC-F", "FC-M", "CxtA-F", "CxtA-M", "CxtB-F", "CxtB-M"]

# Ensure group column exists
df_all["group"] = df_all["context"] + "-" + df_all["sex"]
df_all["group"] = pd.Categorical(df_all["group"], categories=order_6, ordered=True)

# Colors based on sex
palette_6 = [sex_palette[g.split("-")[1]] for g in order_6]

fig, ax = plt.subplots(figsize=(5.2, 3.6), dpi=600)

# Boxplot (white fill, colored outlines by sex)
bp = sns.boxplot(
    data=df_all,
    x="group",
    y="variance",
    order=order_6,
    ax=ax,
    palette=["white"] * 6,
    width=0.6,
    showfliers=False
)

for patch, color in zip(bp.artists, palette_6):
    patch.set_facecolor("white")
    patch.set_edgecolor(color)
    patch.set_linewidth(1.3)

# Stripplot of individual cells
sns.stripplot(
    data=df_all,
    x="group",
    y="variance",
    order=order_6,
    ax=ax,
    palette=palette_6,
    jitter=0.25,
    size=2,
    alpha=0.6,
    edgecolor="white",
    linewidth=0.1
)

# Add context grouping background shading
group_boundaries = {
    "FC":   (0 - 0.5, 1 + 0.5),   # covers FC-F, FC-M
    "CxtA": (2 - 0.5, 3 + 0.5),   # covers CxtA-F, CxtA-M
    "CxtB": (4 - 0.5, 5 + 0.5),   # covers CxtB-F, CxtB-M
}

shade_color = (0.92, 0.92, 0.92)
for i, (ctx, (start, end)) in enumerate(group_boundaries.items()):
    if i % 2 == 0:
        ax.axvspan(start, end, color=shade_color, zorder=0)

# Context labels above (FC, CxtA, CxtB)
y_top = ax.get_ylim()[1]
for ctx in ["FC", "CxtA", "CxtB"]:
    start, end = group_boundaries[ctx]
    midpoint = (start + end) / 2
    ax.text(
        midpoint,
        y_top * 1.03,
        ctx,
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight='bold'
    )

# Label x-axis as sex per bar
sex_xlabels = ["F", "M", "F", "M", "F", "M"]
ax.set_xticklabels(sex_xlabels, rotation=0)

# Remove legend if seaborn created one
if ax.get_legend():
    ax.get_legend().remove()

# Formatting
ax.set_xlabel("")
ax.set_ylabel("Peak Variability Across Sequences")
sns.despine()
plt.tight_layout(pad=2)
plt.rcParams['svg.fonttype'] = 'none'

plt.savefig(
    paths.figure_path("figure3", "individual_cells_FC_A_B_bySex_peak_var.svg"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2731480903.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  bp = sns.boxplot(


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2731480903.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(
C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2731480903.py:88: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(sex_xlabels, rotation=0)


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\2731480903.py:107: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# LMM for 6 groups (sex × context)
import statsmodels.formula.api as smf

# Make sure context and sex are categorical
df_all["context"] = df_all["context"].astype("category")
df_all["sex"] = df_all["sex"].astype("category")

# Fit LMM: variance ~ sex * context + (1 | animal)
model_6 = smf.mixedlm(
    "variance ~ C(sex) * C(context)",
    df_all,
    groups=df_all["animal"]
)

res_6 = model_6.fit(method="lbfgs")
print(res_6.summary())


                  Mixed Linear Model Regression Results
Model:                    MixedLM       Dependent Variable:       variance
No. Observations:         4923          Method:                   REML    
No. Groups:               14            Scale:                    28.8923 
Min. group size:          172           Log-Likelihood:           inf     
Max. group size:          688           Converged:                Yes     
Mean group size:          351.6                                           
--------------------------------------------------------------------------
                               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------
Intercept                      -1.912                                     
C(sex)[T.M]                    -1.343                                     
C(context)[T.CxtB]              0.092    0.435   0.212 0.832 -0.761  0.946
C(context)[T.FC]               -3.660    0.2

C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2054: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2245: UserWarning: The random effects covariance matrix is singular.
  warnings.warn(_warn_cov_sing)
C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:2261: ConvergenceWarnin

In [5]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

tests = [
    # ----- Sex differences within context -----
    ("M - F @ CxtA", [[0, 1, 0, 0, 0, 0]]),
    ("M - F @ CxtB", [[0, 1, 0, 0, 1, 0]]),
    ("M - F @ FC",   [[0, 1, 0, 0, 0, 1]]),

    # ----- Context differences within F -----
    ("F CxtB - CxtA", [[0, 0, 1, 0, 0, 0]]),
    ("F FC - CxtA",   [[0, 0, 0, 1, 0, 0]]),
    ("F CxtB - FC",   [[0, 0, 1, -1, 0, 0]]),

    # ----- Context differences within M -----
    ("M CxtB - CxtA", [[0, 0, 1, 0, 1, 0]]),
    ("M FC - CxtA",   [[0, 0, 0, 1, 0, 1]]),
    ("M CxtB - FC",   [[0, 0, 1, -1, 1, -1]]),

    # ----- Cross-context / cross-sex -----
    ("M CxtA - F CxtB", [[0, 1, -1, 0, 0, 0]]),
    ("M CxtA - F FC",   [[0, 1, 0, -1, 0, 0]]),
    ("M CxtB - F CxtA", [[0, 1, 1, 0, 1, 0]]),
    ("M FC - F CxtA",   [[0, 1, 0, 1, 0, 1]]),
    ("M CxtB - F FC",   [[0, 1, 1, -1, 1, 0]]),
    ("M FC - F CxtB",   [[0, 1, -1, 1, 0, 1]]),
]

# Run Wald tests
pvals = np.array([
    float(res_6.t_test(np.array(L)).pvalue)
    for _, L in tests
])

# Bonferroni across all 15 tests
reject, p_bonf, _, _ = multipletests(pvals, method="bonferroni")

posthoc = pd.DataFrame({
    "contrast": [t[0] for t in tests],
    "p_bonf": p_bonf,
    "sig_0.05": reject
}).sort_values("p_bonf")

posthoc


C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\base\model.py:1678: RuntimeWarning: invalid value encountered in sqrt
  _sd = np.sqrt(self.cov_params(r_matrix=r_matrix, cov_p=cov_p))


,contrast,p_bonf,sig_0.05
8,M CxtB - FC,1.109742e-112,True
4,F FC - CxtA,2.263813e-34,True
6,M CxtB - CxtA,1.333011e-29,True
5,F CxtB - FC,1.155190e-17,True
7,M FC - CxtA,5.183535e-16,True
3,F CxtB - CxtA,1.000000e+00,False
0,M - F @ CxtA,NaN,False
1,M - F @ CxtB,NaN,False
2,M - F @ FC,NaN,False
9,M CxtA - F CxtB,NaN,False


In [6]:
# Cell-level 3-group plot (FC, CxtA, CxtB) and LMM

df_3 = df_all.copy()
df_3["context"] = df_3["context"].astype("category")
df_3["context"] = df_3["context"].cat.reorder_categories(["FC", "CxtA", "CxtB"], ordered=True)

print("Cell counts per context:")
print(df_3.groupby("context")["variance"].count(), "\n")

context_colors = ['grey', '#66C1A6', 'black']

order_3 = ["FC", "CxtA", "CxtB"]
# palette_3 = [context_colors[c] for c in order_3]

fig, ax = plt.subplots(figsize=(3.6, 3.4), dpi=600)

bp = sns.boxplot(
    data=df_3,
    x="context",
    y="variance",
    order=order_3,
    ax=ax,
    palette=["white"] * 3,
    width=0.6,
    showfliers=False
)

for patch, color in zip(bp.artists, context_colors):
    patch.set_facecolor("white")
    patch.set_edgecolor(color)
    patch.set_linewidth(1.3)

sns.stripplot(
    data=df_3,
    x="context",
    y="variance",
    order=order_3,
    ax=ax,
    palette=context_colors,
    dodge=False,
    jitter=0.2,
    size=2,
    alpha=0.6,
    edgecolor="white",
    linewidth=0.3
)

ax.set_xlabel("")
ax.set_ylabel("Peak Variability Across Sequences")
sns.despine(ax=ax)
plt.tight_layout(pad=2)
plt.rcParams['svg.fonttype'] = 'none'

plt.savefig(
    paths.figure_path("figure3", "individual_cells_FC_A_B_peak_var.svg"),
    dpi=600,
    bbox_inches='tight'
)

plt.show()

Cell counts per context:
context
FC      2746
CxtA    1354
CxtB     823
Name: variance, dtype: int64 



C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\3005313190.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_3.groupby("context")["variance"].count(), "\n")
C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\3005313190.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  bp = sns.boxplot(


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\3005313190.py:33: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(


C:\Users\ryansenne\AppData\Local\Temp\ipykernel_38612\3005313190.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
#3N LMM
model = smf.mixedlm(
    "variance ~ C(context)",
    df_3,
    groups=df_3["animal"]
)
res = model.fit(method="lbfgs")
print(res.summary())


C:\Users\ryansenne\PycharmProjects\dCA1_Paper\.venv\lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  variance   
No. Observations:  4923     Method:              REML       
No. Groups:        14       Scale:               29.2744    
Min. group size:   172      Log-Likelihood:      -15319.3850
Max. group size:   688      Converged:           Yes        
Mean group size:   351.6                                    
------------------------------------------------------------
                   Coef. Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept          2.201    0.404  5.450 0.000  1.409  2.992
C(context)[T.CxtA] 2.993    0.192 15.583 0.000  2.617  3.370
C(context)[T.CxtB] 5.560    0.238 23.392 0.000  5.094  6.025
Group Var          2.125    0.161                           



In [8]:
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# Check what parameters your model actually has
print(res.fe_params.index.tolist())

k = len(res.fe_params)
names = list(res.fe_params.index)

def L_from_terms(terms):
    """
    terms: dict like {"C(context)[T.CxtA]": 1, "C(context)[T.CxtB]": -1}
    """
    L = np.zeros((1, k))
    for term, coef in terms.items():
        if term not in names:
            raise ValueError(f"Term '{term}' not in fixed effects: {names}")
        L[0, names.index(term)] = coef
    return L

# Define the 3 pairwise contrasts (assuming FC is the reference)
tests = [
    ("CxtA - FC",   L_from_terms({"C(context)[T.CxtA]": 1})),
    ("CxtB - FC",   L_from_terms({"C(context)[T.CxtB]": 1})),
    ("CxtA - CxtB", L_from_terms({"C(context)[T.CxtA]": 1, "C(context)[T.CxtB]": -1})),
]

pvals = np.array([float(res.t_test(L).pvalue) for _, L in tests])

reject, p_bonf, _, _ = multipletests(pvals, method="bonferroni")

out = pd.DataFrame({
    "contrast": [t[0] for t in tests],
    "p_raw": pvals,
    "p_bonf": p_bonf,
    "sig_0.05_bonf": reject
})

print(out)



['Intercept', 'C(context)[T.CxtA]', 'C(context)[T.CxtB]']
      contrast          p_raw         p_bonf  sig_0.05_bonf
0    CxtA - FC   9.481517e-55   2.844455e-54           True
1    CxtB - FC  5.156967e-121  1.547090e-120           True
2  CxtA - CxtB   1.093312e-19   3.279937e-19           True
